In [ ]:
!pip install transformers datasets sentencepiece accelerate -q

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from datasets import load_dataset

In [ ]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

In [ ]:
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

In [ ]:
small_train = dataset["train"].select(range(5000))
small_val = dataset["validation"].select(range(500))

print(f"Train size: {len(small_train)}")
print(f"Validation size: {len(small_val)}")

In [ ]:
## preprocess /tokenize data
MAX_INPUT = 512
MAX_TARGET = 128

def preprocess(examples):
    inputs = ["summarize: " + doc for doc in examples["article"]]
    model_inputs = tokenizer(
        inputs, 
        max_length = MAX_INPUT,
        truncation = True,
        padding = "max_length",
        
    )
   # with tokenizer.as_target_tokenizer(): // problem kr rha h no attribut aa rh ah
    label_encodings = tokenizer(
    examples["highlights"],
    max_length = MAX_TARGET,
    truncation = True,
    padding = "max_length",
    
)

    model_inputs["labels"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in label_encodings["input_ids"]
    ]
    
    return model_inputs

In [ ]:
##Apply preprocessing
# 
tokenized_train = small_train.map(
    preprocess,
    batched = True,
    remove_columns = small_val.column_names
) 
tokenized_val = small_val.map(
    preprocess,
    batched=True,
    remove_columns= small_val.column_names
)

print("tokenization done!")

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./t2-notes-summerizer",
    num_train_epochs= 3,
    per_device_train_batch_size=4,
     per_device_eval_batch_size=4,
     warmup_steps=200,
     weight_decay= 0.01,
     logging_steps=50,
     eval_strategy="epoch",
     save_strategy="epoch",
     load_best_model_at_end=True,
     fp16=True,
     predict_with_generate=True,
     report_to="none"
)

In [ ]:
# Training setup
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding= True)

trainer = Seq2SeqTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_val,
    processing_class = tokenizer,
    data_collator = data_collator,
)

In [ ]:
# lebels checks
print(type(tokenized_train[0]["labels"]))
print(tokenized_train[0]["labels"][:10])

In [ ]:
trainer.train()

In [ ]:
# save the model
model.save_pretrained("./t5-note-summarizer")
tokenizer.save_pretrained("./t5-note-summarizer")
print("model saved!")

In [ ]:
# test
def summarize(text):
    input_text = "summarize: " + text
    inputs = tokenizer(
        input_text,
        return_tensors = "pt",
        max_length = 512,
        truncation = True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens = 128,
        num_beams = 4,
        early_stopping = True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

my_notes = """
Deep Learning is a specialized area of Machine Learning that focuses on algorithms 
inspired by the structure and function of the human brain, known as Artificial Neural Networks. 
The main idea behind deep learning is to enable computers to learn patterns from large amounts 
of data and make intelligent decisions without being explicitly programmed for every task.
Traditional machine learning methods often require manual feature extraction, where experts 
define the important characteristics of the data. In contrast, deep learning models automatically 
learn useful features from raw data through multiple layers of neural networks. Because these 
models contain many hidden layers, the learning process is called deep learning.
Deep learning has become extremely important in modern artificial intelligence because of its 
ability to process large volumes of data and solve complex problems such as image recognition, 
speech recognition, language translation, and autonomous driving.
The foundation of deep learning is the Artificial Neural Network. Neural networks are 
computational models inspired by biological neural networks found in the human brain. 
A neural network consists of several layers of interconnected nodes called neurons which 
are divided into input layer, hidden layers, and output layer.
"""

print(summarize(my_notes))

In [ ]:
import re

def summary_to_points(text):
    sentences = re.split(r'(?<=[.!?]) +', text.strip())
    points = [f"• {s.strip()}" for s in sentences if s.strip()]
    return "\n".join(points)


result = summarize(my_notes)
points = summary_to_points(result)
print(points)